[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jeevash23/logogram/blob/main/notebooks/ioi-head-patching.ipynb)

# Which heads carry the name? Patching GPT-2 small's heads on IOI

In [ ]:
%pip install logogram

In *When Mary and John went to the store, John gave a drink to*, GPT-2 small answers *Mary*:
the indirect object. Wang et al. (2022, "Interpretability in the Wild: a Circuit for Indirect
Object Identification in GPT-2 small") traced which attention heads do this. This notebook runs
the same kind of experiment with Logogram's Python API: **activation patching**. Each run starts
from the corrupt prompt, where the other name is repeated, copies one head's output in from the
clean prompt, and measures how much of the clean answer comes back.

It uses the IOI example that comes with Logogram, and runs it through the same code as the app
and `logogram run`, so every run lands in the example's project and opens in the app too. GPT-2
small (about 0.5 GB) downloads once; the sweep then takes about a minute on a CPU and seconds on
a GPU.

## Open the example and load the model

In [ ]:
import json

import logogram as lg

# A copy of the example in its own folder (Documents/Logogram/ioi-example), made once.
project = lg.open_example("ioi-gpt2")
# Downloaded once, then checked: Logogram measures that it predicts what the original model does.
model = lg.load_model("openai-community/gpt2")

## The prompts

The dataset holds pairs of prompts that differ in one name. The metric is the logit difference
between the answer and the distractor at the last token: positive where the model prefers the
answer, as it should on the clean prompt, and negative on the corrupt one.

In [ ]:
spec = lg.load_spec(project.root / "experiments/ioi-head-patching/spec.json")
lines = (project.root / spec.dataset.path).read_text(encoding="utf-8").splitlines()
pairs = [json.loads(line) for line in lines]
for pair in pairs[:3]:
    print("clean:  ", pair["clean"])
    print("corrupt:", pair["corrupt"])
    print("answer", repr(pair["answer"]), "· distractor", repr(pair["distractor"]), "\n")
print(len(pairs), "prompt pairs")

## The experiment

The spec states every choice that can change a number. Here: patch clean into corrupt, at every
head, at every position, measured by the logit difference and normalized by the dataset's mean
gap between the clean and corrupt prompts, with a 95% bootstrap interval from 1,000 seeded
resamples.

In [ ]:
print(spec.name)
print(spec.experiment)
print(spec.scope)
print(spec.metric)
print(spec.statistics)

In [ ]:
run = lg.run(spec, project, model=model)
run

## Reading the numbers

Each head's **effect** is the mean, over the prompts, of how far patching it moved the metric,
as a share of the gap between the corrupt and the clean prompt: 1 restores the clean answer
completely, 0 changes nothing, and a negative effect works against the answer.

* The **interval** is a percentile bootstrap interval over prompts: where the mean would fall
  with other prompts like these.
* A sweep tests many heads, and some intervals miss zero by chance. The **band** holds for every
  head at once at the same confidence, and the **q-value** is the false discovery rate at which
  the head would count as a finding (Benjamini and Hochberg, 1995).
* **Flips** count the prompts whose preference for the answer changed sign when the head was
  patched.

In [ ]:
level = round(run.summary["statistics"]["ci"] * 100)
print(f"{'head':<8}{'effect':>8}  {level}% interval{'':<5}simultaneous band{'':<4}q{'':<8}flips")
for site in run.strongest(8):
    e, band = site["effect"], site["band"]
    print(
        f"{site['label']:<8}{e['mean']:>+8.3f}  [{e['lo']:+.3f}, {e['hi']:+.3f}]  "
        f"[{band['lo']:+.3f}, {band['hi']:+.3f}]  {site['q']:<8.2g} {site['sign_flips']}/{site['n']}"
    )

## One head, prompt by prompt

A mean can hide prompts that go the other way. Every run keeps each prompt's value, and the
statistics are computed from them.

In [ ]:
top = run.strongest(1)[0]
evidence = run.site(top["index"])
by_prompt = {p["index"]: p for p in evidence["prompts"]}
effects = sorted(p["effect"] for p in evidence["prompts"])
print(f"{top['label']}: {len(effects)} prompts, from {effects[0]:+.2f} to {effects[-1]:+.2f}")
print(f"{top['opposite_sign']} prompts go against the mean\n")
for title, indices in (("Moved most", evidence["strongest"][:3]), ("Moved least", evidence["weakest"][:3])):
    print(title)
    for i in indices:
        print(f"  {by_prompt[i]['effect']:+.2f}  {by_prompt[i]['corrupt']}")

## Does it hold the other way round?

Patching clean into corrupt asks which heads are enough to *restore* the answer. Patching
corrupt into clean asks which heads the answer *needs*: removing one breaks it. The two can
disagree, for instance when backup heads take over. `check_robustness` reruns the sweep with one
choice changed, and `compare_runs` pairs the two runs prompt by prompt.

In [ ]:
noising = lg.check_robustness(
    run, experiment={"kind": "activation_patching", "direction": "corrupt_to_clean"}
)
comparison = lg.compare_runs(run, noising)
print(f"Rank correlation {comparison['spearman']:.2f};", end=" ")
print(f"{comparison['top_overlap']} of the {comparison['top_k']} strongest heads are strongest in both.")
print(f"\n{'head':<8}{'restores':>9}{'needed':>9}")
for change in comparison["changes"][:10]:
    print(
        f"{change['label']:<8}{change['effect_a']['mean']:>+9.3f}{change['effect_b']['mean']:>+9.3f}  "
        + " ".join(change["flags"])
    )

## Write it up

The methods paragraph says every choice in sentences, and the report is one HTML file anyone can
open without Logogram. Opening the project in the app (`logogram open` and the project's folder)
shows these runs in its history.

In [ ]:
print(run.methods())
run.report("ioi-heads.html")
print("Wrote ioi-heads.html")